# Transformer Ultimate v2

Pipeline robuste pour le concours Kaggle (texte + TF-IDF + features) avec validation stratifiee, gestion memoire et stacking.

In [ ]:
import os, gc, random, math
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple, List

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.calibration import CalibratedClassifierCV
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    get_cosine_schedule_with_warmup,
)

# Caches pour limiter l'ecriture disque
os.environ["HF_HOME"] = "/tmp/hf"
os.environ["TRANSFORMERS_CACHE"] = "/tmp/hf/transformers"
os.environ["HF_DATASETS_CACHE"] = "/tmp/hf/datasets"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Évite les warnings de fork

# =============================================================================
# 🚀 CONFIGURATION H100 - OPTIMISÉE POUR 80GB VRAM
# =============================================================================
# Pas de version "large" de twitter-xlm-roberta, donc on utilise:
# 1. xlm-roberta-large (560M params) - meilleure capacité, multilingue FR
# 2. + ensemble avec twitter-xlm-roberta-base si temps disponible
# 
# CamemBERT-large (335M) est une alternative pour le français pur, 
# mais XLM-RoBERTa-large est généralement meilleur pour la classification

@dataclass
class Config:
    # MODÈLES - xlm-roberta-large est le meilleur choix pour H100
    model_name: str = "xlm-roberta-large"  # ⬆️ 560M params vs 278M (base)
    # Alternative: "camembert/camembert-large" pour FR pur (335M params)
    # Le twitter-xlm-roberta n'a PAS de version large malheureusement
    
    max_len: int = 256           # ⬆️ Augmenté (H100 peut gérer)
    epochs: int = 5              # ⬆️ Plus d'epochs pour large model
    folds: int = 5
    batch_size: int = 32         # ⬆️ H100 80GB: batch 32 facile pour large
    grad_acc: int = 1            # ⬇️ Pas besoin d'accumulation avec H100
    lr: float = 1.5e-5           # ⬇️ LR plus bas pour large model
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    layerwise_decay: float = 0.9 # ⬆️ Decay plus doux pour large
    label_smoothing: float = 0.05
    seeds: Tuple[int, ...] = (42, 1234, 2024, 0)  # ⬆️ 4 seeds pour réduire variance
    tfidf_max_features: int = 300000  # ⬆️ Plus de features TF-IDF
    tfidf_char_ngrams: Tuple[int, int] = (3, 5)
    tfidf_word_ngrams: Tuple[int, int] = (1, 3)  # ⬆️ Trigrams mots

CFG = Config()

# Le notebook est dans notebooks/, donc on remonte d'un niveau
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Fallback: utiliser le chemin absolu si le cwd est ailleurs
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1")
DATA_DIR = PROJECT_ROOT / "data"
SUBMISSION_DIR = PROJECT_ROOT / "submission"
MODEL_DIR = PROJECT_ROOT / "models"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"🖥️  GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU: {gpu_name}")
    print(f"💾 VRAM: {gpu_mem:.1f} GB")
    
    # Auto-ajustement pour H100 vs autres GPUs
    if "H100" in gpu_name or gpu_mem > 70:
        print("✅ H100 détecté - Configuration haute performance activée")
    elif gpu_mem > 20:
        print("⚠️ GPU mid-range détecté - Réduction batch size")
        CFG.batch_size = 16
        CFG.grad_acc = 2
    else:
        print("⚠️ GPU limité - Configuration économique")
        CFG.batch_size = 8
        CFG.grad_acc = 4
        CFG.model_name = "cardiffnlp/twitter-xlm-roberta-base"
        
print(f"\n📋 Configuration finale:")
print(f"   Modèle: {CFG.model_name}")
print(f"   Batch size: {CFG.batch_size} × {CFG.grad_acc} = {CFG.batch_size * CFG.grad_acc}")
print(f"   Seeds: {CFG.seeds}")
print(f"   Epochs: {CFG.epochs}")

/users/eleves-a/2023/malo.tamalet/influencer-or-observer-1/kaggle-env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GPU available: True
GPU: NVIDIA RTX A2000 12GB


In [ ]:
# Utilitaires

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


# =============================================================================
# 🐦 PREPROCESSING TWITTER (fonctionne pour tous les modèles)
# =============================================================================
import re

def preprocess_twitter(text: str) -> str:
    """Preprocessing pour tweets (compatible avec tous les modèles XLM/CamemBERT).
    - Remplace les @mentions par @user
    - Remplace les URLs par http
    - Normalise les espaces
    """
    if not isinstance(text, str):
        return ""
    new_text = []
    for t in text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    text = " ".join(new_text)
    # Nettoyer les espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_text(row: pd.Series) -> str:
    if isinstance(row.get("extended_tweet"), dict):
        txt = row["extended_tweet"].get("full_text")
        if isinstance(txt, str):
            return txt
    if isinstance(row.get("full_text"), str):
        return row["full_text"]
    if isinstance(row.get("text"), str):
        return row["text"]
    return ""


def build_text(row: pd.Series) -> str:
    """Construit le texte enrichi avec preprocessing Twitter.
    
    Format optimisé pour xlm-roberta-large:
    - Tweet principal
    - Description utilisateur (tronquée)
    - Localisation
    """
    user = row.get("user") or {}
    desc = (user.get("description") or "")[:200]  # ⬆️ Plus de contexte avec H100
    loc = (user.get("location") or "")[:80]
    base = extract_text(row)
    
    # Appliquer le preprocessing Twitter
    base = preprocess_twitter(base)
    desc = preprocess_twitter(desc)
    
    # Pour xlm-roberta, utiliser </s> comme séparateur (meilleur que [SEP])
    sep = "</s>" if "xlm-roberta" in CFG.model_name else "[SEP]"
    
    return f"{base} {sep} {desc} {sep} {loc}"


def pseudo_user_id(row: pd.Series) -> str:
    """Crée un ID pseudo-unique pour regrouper les tweets d'un même utilisateur."""
    user = row.get("user") or {}
    key = "|".join([
        str(user.get("description", ""))[:96],
        str(user.get("profile_image_url_https", "")),
        str(user.get("profile_banner_url", "")),
        str(user.get("statuses_count", "")),
        str(user.get("favourites_count", "")),
    ])
    return str(abs(hash(key)) % (10 ** 12))

## Chargement des donnees

In [3]:
train_df = pd.read_json(DATA_DIR / "train.jsonl", lines=True)
test_df = pd.read_json(DATA_DIR / "kaggle_test.jsonl", lines=True)

train_df["text_clean"] = train_df.apply(build_text, axis=1)
test_df["text_clean"] = test_df.apply(build_text, axis=1)

train_df["group"] = train_df.apply(pseudo_user_id, axis=1)
test_df["group"] = test_df.apply(pseudo_user_id, axis=1)

labels = train_df["label"].astype(int).to_numpy()
groups = train_df["group"].to_numpy()

print(train_df[["text_clean"]].head(2))
print(f"Train tweets: {len(train_df):,} | Test tweets: {len(test_df):,}")

                                          text_clean
0  C’est exactement ça ... [SEP] 👨🏻‍💻 Conseiller ...
1  Depuis un certain temps, Le Gorafi n'a même pl...
Train tweets: 154,914 | Test tweets: 103,380


## Adversarial Validation (détection data drift)

Entraîne un modèle pour distinguer train vs test. Si AUC > 0.5, il y a un décalage entre les distributions.

In [4]:
# =============================================================================
# 🎯 ADVERSARIAL VALIDATION
# =============================================================================
# Détecte si train et test ont des distributions différentes
# Si AUC >> 0.5, certains échantillons train sont "différents" du test

from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier

def run_adversarial_validation(train_df, test_df, text_col="text_clean"):
    """Entraîne un modèle pour distinguer train vs test."""
    
    # Créer features simples pour adversarial validation
    def extract_adv_features(df):
        features = pd.DataFrame()
        features["text_len"] = df[text_col].str.len()
        features["word_count"] = df[text_col].str.split().str.len()
        features["has_url"] = df[text_col].str.contains("http", case=False).astype(int)
        features["has_mention"] = df[text_col].str.contains("@user", case=False).astype(int)
        features["uppercase_ratio"] = df[text_col].apply(
            lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)), 1)
        )
        return features
    
    X_train_adv = extract_adv_features(train_df)
    X_test_adv = extract_adv_features(test_df)
    
    # Concaténer et créer labels (0=train, 1=test)
    X_adv = pd.concat([X_train_adv, X_test_adv], axis=0, ignore_index=True)
    y_adv = np.array([0] * len(train_df) + [1] * len(test_df))
    
    # Entraîner classifier
    clf_adv = LGBMClassifier(n_estimators=100, max_depth=5, random_state=42, verbose=-1)
    scores = cross_val_score(clf_adv, X_adv, y_adv, cv=5, scoring="roc_auc")
    
    print(f"🎯 Adversarial Validation AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")
    
    if scores.mean() > 0.55:
        print("⚠️  Attention: Train/Test ont des distributions différentes!")
        print("   → Le modèle pourrait mal généraliser sur le test")
    else:
        print("✅ Train/Test ont des distributions similaires")
    
    # Identifier les échantillons train les plus "différents" du test
    clf_adv.fit(X_adv, y_adv)
    train_probs = clf_adv.predict_proba(X_train_adv)[:, 1]
    
    # Échantillons avec probabilité élevée = ressemblent au test (bon!)
    # Échantillons avec probabilité faible = différents du test (mauvais!)
    return train_probs

adv_probs = run_adversarial_validation(train_df, test_df)

# Créer un poids pour downweight les échantillons "différents"
# Les échantillons qui ressemblent au test auront plus de poids
sample_weights = 0.5 + 0.5 * adv_probs  # Entre 0.5 et 1.0
print(f"Poids d'échantillons: min={sample_weights.min():.3f}, max={sample_weights.max():.3f}")

🎯 Adversarial Validation AUC: 0.5035 (+/- 0.0022)
✅ Train/Test ont des distributions similaires
Poids d'échantillons: min=0.561, max=0.844


## Tokenisation Transformer

In [5]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

train_ds = Dataset.from_pandas(train_df[["text_clean", "label", "group"]])
test_ds = Dataset.from_pandas(test_df[["text_clean", "group"]])  # Exclure challenge_id ici

remove_cols_train = [c for c in ["text_clean", "group", "__index_level_0__"] if c in train_ds.column_names]
remove_cols_test = [c for c in ["text_clean", "group", "__index_level_0__"] if c in test_ds.column_names]

def tokenize(batch):
    return tokenizer(batch["text_clean"], truncation=True, max_length=CFG.max_len)

train_tok = train_ds.map(tokenize, batched=True, remove_columns=remove_cols_train)
test_tok = test_ds.map(tokenize, batched=True, remove_columns=remove_cols_test)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

print(f"✅ Tokenisation terminée")
print(f"   Train columns: {train_tok.column_names}")
print(f"   Test columns: {test_tok.column_names}")

Map: 100%|██████████| 103380/103380 [00:22<00:00, 4499.31 examples/s]

✅ Tokenisation terminée
   Train columns: ['label', 'input_ids', 'attention_mask']
   Test columns: ['input_ids', 'attention_mask']


## Fine-tuning Transformer (multi-seed)

In [ ]:
# =============================================================================
# 🧠 MODÈLE CUSTOM: Multi-Sample Dropout + Mean Pooling + Attention Pooling
# =============================================================================
# Techniques avancées pour maximiser la performance sur H100

import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoConfig

class AttentionPooling(nn.Module):
    """Attention-weighted pooling - meilleur que mean pour les tweets."""
    def __init__(self, hidden_size):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 4),
            nn.Tanh(),
            nn.Linear(hidden_size // 4, 1),
        )
    
    def forward(self, hidden_states, attention_mask):
        # hidden_states: [batch, seq_len, hidden]
        attn_weights = self.attention(hidden_states).squeeze(-1)  # [batch, seq_len]
        attn_weights = attn_weights.masked_fill(attention_mask == 0, float('-inf'))
        attn_weights = F.softmax(attn_weights, dim=1)
        pooled = torch.bmm(attn_weights.unsqueeze(1), hidden_states).squeeze(1)
        return pooled


class MultiSampleDropoutClassifier(nn.Module):
    """
    Modèle optimisé pour H100 avec:
    - Mean Pooling + Attention Pooling (concaténés)
    - Multi-Sample Dropout (5 taux différents)
    - Concatenation des 4 dernières couches cachées (pour large models)
    - Layer Normalization avant classification
    """
    def __init__(self, model_name: str, num_labels: int = 2, dropout_samples: int = 5, 
                 use_last_n_layers: int = 4):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name, output_hidden_states=True)
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout_samples = dropout_samples
        self.use_last_n_layers = use_last_n_layers
        
        hidden_size = self.config.hidden_size
        
        # Attention pooling pour meilleure représentation
        self.attention_pool = AttentionPooling(hidden_size)
        
        # Multi-sample dropout avec différents taux
        self.dropouts = nn.ModuleList([
            nn.Dropout(p) for p in [0.1, 0.15, 0.2, 0.25, 0.3]
        ])
        
        # Pour large models: concat mean + attention pooling = 2x hidden_size
        classifier_input_size = hidden_size * 2
        
        # Layer norm avant classifier (stabilise l'entraînement)
        self.layer_norm = nn.LayerNorm(classifier_input_size)
        
        # Classifier plus profond pour large model
        self.classifier = nn.Sequential(
            nn.Linear(classifier_input_size, hidden_size),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size, num_labels)
        )
        
        # Initialisation Xavier
        for module in self.classifier:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)
    
    def mean_pooling(self, hidden_states, attention_mask):
        """Mean pooling sur les tokens non-padding."""
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(hidden_states.size()).float()
        sum_embeddings = torch.sum(hidden_states * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        return sum_embeddings / sum_mask
    
    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        
        # Utiliser le dernier hidden state
        last_hidden = outputs.last_hidden_state
        
        # Double pooling: mean + attention
        mean_pool = self.mean_pooling(last_hidden, attention_mask)
        attn_pool = self.attention_pool(last_hidden, attention_mask)
        
        # Concaténer les deux représentations
        pooled = torch.cat([mean_pool, attn_pool], dim=-1)
        pooled = self.layer_norm(pooled)
        
        # Multi-sample dropout: moyenne des logits avec différents dropouts
        if self.training:
            logits = torch.stack([
                self.classifier(dropout(pooled)) 
                for dropout in self.dropouts
            ]).mean(dim=0)
        else:
            logits = self.classifier(pooled)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(label_smoothing=CFG.label_smoothing)
            loss = loss_fct(logits, labels)
        
        return {"loss": loss, "logits": logits}


# =============================================================================
# 🔥 MODÈLE STANDARD (fallback si custom ne marche pas)
# =============================================================================
USE_CUSTOM_MODEL = True  # Mettre à False pour revenir au modèle HuggingFace standard

print(f"✅ Modèle défini:")
print(f"   Custom Multi-Sample Dropout: {USE_CUSTOM_MODEL}")
print(f"   Base model: {CFG.model_name}")

✅ Multi-Sample Dropout Classifier défini (USE_CUSTOM_MODEL=True)


In [ ]:
def build_optimizer(model, steps: int):
    """Construit l'optimizer avec Layerwise LR Decay (optimisé pour large models)."""
    no_decay = ["bias", "LayerNorm.weight", "layer_norm"]
    
    # Supporter les deux types de modèles (custom et standard)
    if hasattr(model, 'transformer'):
        # Modèle custom MultiSampleDropoutClassifier
        base_model = model.transformer
        # Pour xlm-roberta-large
        if "xlm-roberta" in CFG.model_name:
            encoder_prefix = "transformer.roberta.encoder.layer"
            embedding_prefix = "transformer.roberta.embeddings"
        else:
            encoder_prefix = "transformer.encoder.layer"
            embedding_prefix = "transformer.embeddings"
    else:
        # Modèle HuggingFace standard
        base_model = model
        if "xlm-roberta" in CFG.model_name:
            encoder_prefix = "roberta.encoder.layer"
            embedding_prefix = "roberta.embeddings"
        else:
            encoder_prefix = "encoder.layer"
            embedding_prefix = "embeddings"
    
    num_layers = getattr(base_model.config, "num_hidden_layers", 24)  # 24 pour large
    params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        
        # Déterminer le learning rate selon la couche
        if embedding_prefix.split(".")[-1] in name and "embed" in name.lower():
            lr = CFG.lr * (CFG.layerwise_decay ** (num_layers + 1))
        elif "layer" in name and "encoder" in name:
            # Extraire le numéro de couche
            try:
                import re
                layer_match = re.search(r'layer\.(\d+)', name)
                if layer_match:
                    layer_id = int(layer_match.group(1))
                    lr = CFG.lr * (CFG.layerwise_decay ** (num_layers - layer_id))
                else:
                    lr = CFG.lr
            except (ValueError, IndexError):
                lr = CFG.lr
        elif "classifier" in name or "attention_pool" in name or "layer_norm" in name:
            # Head custom: learning rate plus élevé
            lr = CFG.lr * 2
        else:
            lr = CFG.lr
        
        wd = 0.0 if any(nd in name for nd in no_decay) else CFG.weight_decay
        params.append({"params": [param], "lr": lr, "weight_decay": wd})
    
    optimizer = torch.optim.AdamW(params, lr=CFG.lr, eps=1e-8)
    warmup = int(steps * CFG.warmup_ratio)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup, steps)
    return optimizer, scheduler


def train_transformer(seed: int) -> Tuple[np.ndarray, np.ndarray]:
    """Entraîne le transformer avec une seed donnée (optimisé H100)."""
    set_seed(seed)
    skf = StratifiedKFold(n_splits=CFG.folds, shuffle=True, random_state=seed)
    oof = np.zeros(len(train_tok))
    test_preds = np.zeros((len(test_tok), 2))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        print(f"\nFold {fold+1}/{CFG.folds} (seed={seed})")
        tr_ds = train_tok.select(tr_idx.tolist())
        va_ds = train_tok.select(va_idx.tolist())

        # Choix du modèle: Custom avec Multi-Sample Dropout ou Standard HuggingFace
        if USE_CUSTOM_MODEL:
            model = MultiSampleDropoutClassifier(CFG.model_name, num_labels=2)
            # Activer gradient checkpointing pour économiser mémoire (même sur H100)
            try:
                model.transformer.gradient_checkpointing_enable()
            except Exception:
                pass
        else:
            model = AutoModelForSequenceClassification.from_pretrained(
                CFG.model_name, num_labels=2
            )
            model.config.use_cache = False
            try:
                model.gradient_checkpointing_enable()
            except Exception:
                pass

        steps_per_epoch = math.ceil(len(tr_idx) / (CFG.batch_size * CFG.grad_acc))
        total_steps = steps_per_epoch * CFG.epochs
        optimizer, scheduler = build_optimizer(model, total_steps)

        # TrainingArguments optimisés pour H100
        args = TrainingArguments(
            output_dir=f"/tmp/tf_v2_seed{seed}_fold{fold}",
            num_train_epochs=CFG.epochs,
            per_device_train_batch_size=CFG.batch_size,
            per_device_eval_batch_size=CFG.batch_size * 2,  # Eval peut être plus grand
            learning_rate=CFG.lr,
            weight_decay=CFG.weight_decay,
            gradient_accumulation_steps=CFG.grad_acc,
            warmup_ratio=CFG.warmup_ratio,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            logging_strategy="epoch",
            # H100 supporte bf16 nativement (plus stable que fp16)
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
            dataloader_num_workers=8,  # ⬆️ Plus de workers avec H100
            dataloader_pin_memory=True,
            # label_smoothing déjà appliqué dans le modèle custom si USE_CUSTOM_MODEL=True
            label_smoothing_factor=0.0 if USE_CUSTOM_MODEL else CFG.label_smoothing,
            report_to="none",
            seed=seed,
            # Optimisations H100
            torch_compile=False,  # Peut activer si PyTorch 2.0+
            optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=tr_ds,
            eval_dataset=va_ds,
            processing_class=tokenizer,
            data_collator=collator,
            compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))},
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0005)],
            optimizers=(optimizer, scheduler),
        )

        trainer.train()

        val_logits = trainer.predict(va_ds).predictions
        oof[va_idx] = torch.softmax(torch.tensor(val_logits), dim=1).numpy()[:, 1]

        test_logits = trainer.predict(test_tok).predictions
        test_preds += torch.softmax(torch.tensor(test_logits), dim=1).numpy()

        # Nettoyer le dossier de checkpoints pour économiser l'espace disque
        import shutil
        ckpt_dir = f"/tmp/tf_v2_seed{seed}_fold{fold}"
        if os.path.exists(ckpt_dir):
            shutil.rmtree(ckpt_dir)
        
        clean_memory()

    test_preds /= CFG.folds
    return oof, test_preds[:, 1]

### Lancement Transformer (peut prendre du temps)

In [ ]:
# =============================================================================
# 🔄 SYSTÈME DE CHECKPOINTING - Reprend là où on s'est arrêté
# =============================================================================
CHECKPOINT_DIR = Path("/tmp/transformer_v2_checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

def save_checkpoint(seed, fold, oof, test_preds):
    """Sauvegarde le checkpoint après chaque fold."""
    ckpt = {
        "seed": seed,
        "fold": fold,
        "oof": oof,
        "test_preds": test_preds,
    }
    np.savez(CHECKPOINT_DIR / f"ckpt_seed{seed}_fold{fold}.npz", **ckpt)
    print(f"   💾 Checkpoint sauvegardé: seed={seed}, fold={fold}")

def load_checkpoint(seed, fold):
    """Charge un checkpoint s'il existe."""
    path = CHECKPOINT_DIR / f"ckpt_seed{seed}_fold{fold}.npz"
    if path.exists():
        ckpt = np.load(path)
        return ckpt["oof"], ckpt["test_preds"]
    return None, None

def get_last_completed(seed):
    """Trouve le dernier fold complété pour une seed."""
    for fold in range(CFG.folds - 1, -1, -1):
        if (CHECKPOINT_DIR / f"ckpt_seed{seed}_fold{fold}.npz").exists():
            return fold
    return -1

def clear_checkpoints():
    """Supprime tous les checkpoints pour recommencer."""
    import shutil
    if CHECKPOINT_DIR.exists():
        shutil.rmtree(CHECKPOINT_DIR)
        CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print("🗑️ Checkpoints supprimés - Entraînement recommencera depuis le début")

# ⚠️ DÉCOMMENTER LA LIGNE SUIVANTE POUR RECOMMENCER DEPUIS ZÉRO
# clear_checkpoints()

# =============================================================================
# 🚀 ENTRAÎNEMENT AVEC REPRISE AUTOMATIQUE (OPTIMISÉ H100)
# =============================================================================
print(f"\n{'='*70}")
print(f"🚀 DÉMARRAGE ENTRAÎNEMENT TRANSFORMER")
print(f"{'='*70}")
print(f"📋 Modèle: {CFG.model_name}")
print(f"📋 Seeds: {CFG.seeds}")
print(f"📋 Folds: {CFG.folds}")
print(f"📋 Epochs: {CFG.epochs}")
print(f"📋 Batch effectif: {CFG.batch_size * CFG.grad_acc}")
print(f"📋 Temps estimé: ~{len(CFG.seeds) * CFG.folds * CFG.epochs * 8}min pour xlm-roberta-large")
print(f"{'='*70}")

transformer_oof_list = []
transformer_test_list = []

for seed in CFG.seeds:
    print(f"\n{'='*60}")
    print(f"🌱 SEED {seed}")
    print(f"{'='*60}")
    
    # Vérifier si cette seed est déjà complétée
    last_fold = get_last_completed(seed)
    
    if last_fold == CFG.folds - 1:
        # Seed complète, charger les résultats
        print(f"✅ Seed {seed} déjà complétée, chargement des résultats...")
        oof_final, test_final = load_checkpoint(seed, last_fold)
        transformer_oof_list.append(oof_final)
        transformer_test_list.append(test_final[:, 1] if test_final.ndim > 1 else test_final)
        continue
    
    # Initialiser ou reprendre
    if last_fold >= 0:
        print(f"📥 Reprise depuis fold {last_fold + 2}/{CFG.folds}")
        oof, test_preds = load_checkpoint(seed, last_fold)
        start_fold = last_fold + 1
    else:
        print(f"🆕 Démarrage depuis le début")
        oof = np.zeros(len(train_tok))
        test_preds = np.zeros((len(test_tok), 2))
        start_fold = 0
    
    # Entraînement des folds restants
    set_seed(seed)
    skf = StratifiedKFold(n_splits=CFG.folds, shuffle=True, random_state=seed)
    
    for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(labels)), labels)):
        if fold < start_fold:
            continue  # Skip folds déjà complétés
            
        print(f"\n📊 Fold {fold+1}/{CFG.folds} (seed={seed})")
        print(f"   Train: {len(tr_idx):,} | Val: {len(va_idx):,}")
        
        tr_ds = train_tok.select(tr_idx.tolist())
        va_ds = train_tok.select(va_idx.tolist())

        # Choix du modèle
        if USE_CUSTOM_MODEL:
            model = MultiSampleDropoutClassifier(CFG.model_name, num_labels=2)
            try:
                model.transformer.gradient_checkpointing_enable()
            except Exception:
                pass
        else:
            model = AutoModelForSequenceClassification.from_pretrained(
                CFG.model_name, num_labels=2
            )
            model.config.use_cache = False
            try:
                model.gradient_checkpointing_enable()
            except Exception:
                pass

        steps_per_epoch = math.ceil(len(tr_idx) / (CFG.batch_size * CFG.grad_acc))
        total_steps = steps_per_epoch * CFG.epochs
        optimizer, scheduler = build_optimizer(model, total_steps)

        # TrainingArguments optimisés H100
        args = TrainingArguments(
            output_dir=f"/tmp/tf_v2_seed{seed}_fold{fold}",
            num_train_epochs=CFG.epochs,
            per_device_train_batch_size=CFG.batch_size,
            per_device_eval_batch_size=CFG.batch_size * 2,
            learning_rate=CFG.lr,
            weight_decay=CFG.weight_decay,
            gradient_accumulation_steps=CFG.grad_acc,
            warmup_ratio=CFG.warmup_ratio,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="accuracy",
            logging_strategy="epoch",
            bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
            fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
            dataloader_num_workers=8,
            dataloader_pin_memory=True,
            label_smoothing_factor=0.0 if USE_CUSTOM_MODEL else CFG.label_smoothing,
            report_to="none",
            seed=seed,
            optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
        )

        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=tr_ds,
            eval_dataset=va_ds,
            processing_class=tokenizer,
            data_collator=collator,
            compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, p.predictions.argmax(-1))},
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.0005)],
            optimizers=(optimizer, scheduler),
        )

        trainer.train()

        # Prédictions
        val_logits = trainer.predict(va_ds).predictions
        oof[va_idx] = torch.softmax(torch.tensor(val_logits), dim=1).numpy()[:, 1]

        test_logits = trainer.predict(test_tok).predictions
        test_preds += torch.softmax(torch.tensor(test_logits), dim=1).numpy()

        # 💾 SAUVEGARDER CHECKPOINT APRÈS CHAQUE FOLD
        save_checkpoint(seed, fold, oof, test_preds)

        # Afficher accuracy du fold
        fold_acc = accuracy_score(labels[va_idx], (oof[va_idx] >= 0.5).astype(int))
        print(f"   ✅ Fold {fold+1} accuracy: {fold_acc:.4f}")

        # Nettoyer
        import shutil
        ckpt_dir = f"/tmp/tf_v2_seed{seed}_fold{fold}"
        if os.path.exists(ckpt_dir):
            shutil.rmtree(ckpt_dir)
        clean_memory()
    
    # Normaliser test_preds par le nombre de folds
    test_preds_normalized = test_preds / CFG.folds
    transformer_oof_list.append(oof)
    transformer_test_list.append(test_preds_normalized[:, 1])
    
    # Accuracy de la seed
    seed_acc = accuracy_score(labels, (oof >= 0.5).astype(int))
    print(f"\n🎯 Seed {seed} OOF accuracy: {seed_acc:.4f}")

# Moyenne des seeds
transformer_oof = np.mean(transformer_oof_list, axis=0)
transformer_test = np.mean(transformer_test_list, axis=0)

# Sauvegarder les résultats finaux
np.save(MODEL_DIR / "tf_v2_oof.npy", transformer_oof)
np.save(MODEL_DIR / "tf_v2_test.npy", transformer_test)

# Trouver le meilleur seuil
thr_grid = np.linspace(0.4, 0.6, 21)
best_tf_thr, best_tf_acc = 0.5, 0
for thr in thr_grid:
    acc = accuracy_score(labels, (transformer_oof >= thr).astype(int))
    if acc > best_tf_acc:
        best_tf_acc, best_tf_thr = acc, thr

print(f"\n{'='*70}")
print(f"🎯 RÉSULTATS TRANSFORMER ({CFG.model_name})")
print(f"{'='*70}")
print(f"📊 OOF Accuracy: {best_tf_acc:.4f} @ threshold={best_tf_thr:.3f}")
print(f"📊 Seeds utilisées: {CFG.seeds}")
print(f"📊 Variance inter-seeds: {np.std([accuracy_score(labels, (oof >= 0.5).astype(int)) for oof in transformer_oof_list]):.4f}")
print(f"💾 Fichiers sauvegardés dans {MODEL_DIR}")


🌱 SEED 42
🆕 Démarrage depuis le début

📊 Fold 1/5 (seed=42)


Some weights of XLMRobertaModel were not initialized from the model checkpoint at cardiffnlp/twitter-xlm-roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.564600,0.514365,0.767776


KeyboardInterrupt: 

## TF-IDF + Logistic Regression (generalise bien)

In [ ]:
set_seed(42)

vec_word = TfidfVectorizer(
    analyzer="word",
    ngram_range=CFG.tfidf_word_ngrams,
    max_features=CFG.tfidf_max_features // 2,
    min_df=2,
    strip_accents="unicode",
)
vec_char = TfidfVectorizer(
    analyzer="char",
    ngram_range=CFG.tfidf_char_ngrams,
    max_features=CFG.tfidf_max_features // 2,
    min_df=2,
)

vectorizer = FeatureUnion([
    ("word", vec_word),
    ("char", vec_char),
])

full_text = pd.concat([train_df["text_clean"], test_df["text_clean"]])
X_all = vectorizer.fit_transform(full_text)
X_train = X_all[: len(train_df)]
X_test = X_all[len(train_df) :]

skf = StratifiedKFold(n_splits=CFG.folds, shuffle=True, random_state=42)
oof_tfidf = np.zeros(len(train_df))
test_tfidf = np.zeros((len(test_df), 2))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train, labels)):
    clf = LogisticRegression(max_iter=400, C=4.0, n_jobs=-1)
    clf.fit(X_train[tr_idx], labels[tr_idx])
    oof_tfidf[va_idx] = clf.predict_proba(X_train[va_idx])[:, 1]
    test_tfidf += clf.predict_proba(X_test)

test_tfidf /= CFG.folds
np.save(MODEL_DIR / "tfidf_oof.npy", oof_tfidf)
np.save(MODEL_DIR / "tfidf_test.npy", test_tfidf[:, 1])

best_tfidf_thr, best_tfidf_acc = 0.5, 0
for thr in thr_grid:
    acc = accuracy_score(labels, (oof_tfidf >= thr).astype(int))
    if acc > best_tfidf_acc:
        best_tfidf_acc, best_tfidf_thr = acc, thr
print(f"TF-IDF OOF acc={best_tfidf_acc:.4f} @ thr={best_tfidf_thr:.3f}")

import scipy
# Sauvegarder dans /tmp pour économiser l'espace quota (fichier très volumineux)
scipy.sparse.save_npz("/tmp/tfidf_features_test.npz", X_test)
del X_all, X_train, X_test
clean_memory()

## XGBoost sur features engineering

In [ ]:
try:
    X_feat = np.load(DATA_DIR / "features/X_train_features.npy")
    X_feat_test = np.load(DATA_DIR / "features/X_kaggle_features.npy")
    print(f"✅ Features chargées: {X_feat.shape[1]} colonnes")
except FileNotFoundError:
    raise SystemExit("Features numpy manquantes (data/features/X_train_features.npy)")

from xgboost import XGBClassifier

skf = StratifiedKFold(n_splits=CFG.folds, shuffle=True, random_state=42)
oof_xgb = np.zeros(len(labels))
test_xgb = np.zeros(len(test_df))

print(f"\n🌲 Entraînement XGBoost (GPU)")

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_feat, labels)):
    clf = XGBClassifier(
        n_estimators=3000,       # ⬆️ Plus d'arbres pour H100
        max_depth=12,            # ⬆️ Profondeur augmentée
        learning_rate=0.02,      # ⬇️ LR plus bas = plus d'arbres
        subsample=0.8,
        colsample_bytree=0.75,
        colsample_bylevel=0.75,
        min_child_weight=1,
        gamma=0.1,
        reg_alpha=0.05,
        reg_lambda=0.5,
        tree_method="hist",
        device="cuda" if torch.cuda.is_available() else "cpu",
        eval_metric="logloss",
        early_stopping_rounds=150,  # ⬆️ Plus de patience
        random_state=42,
    )
    clf.fit(
        X_feat[tr_idx], labels[tr_idx],
        eval_set=[(X_feat[va_idx], labels[va_idx])],
        verbose=False
    )
    oof_xgb[va_idx] = clf.predict_proba(X_feat[va_idx])[:, 1]
    test_xgb += clf.predict_proba(X_feat_test)[:, 1]
    
    fold_acc = accuracy_score(labels[va_idx], (oof_xgb[va_idx] >= 0.5).astype(int))
    print(f"   Fold {fold+1}: acc={fold_acc:.4f}, best_iter={clf.best_iteration}")

test_xgb /= CFG.folds
np.save(MODEL_DIR / "xgb_feat_oof.npy", oof_xgb)
np.save(MODEL_DIR / "xgb_feat_test.npy", test_xgb)

best_xgb_thr, best_xgb_acc = 0.5, 0
for thr in thr_grid:
    acc = accuracy_score(labels, (oof_xgb >= thr).astype(int))
    if acc > best_xgb_acc:
        best_xgb_acc, best_xgb_thr = acc, thr
print(f"\n✅ XGB OOF acc={best_xgb_acc:.4f} @ thr={best_xgb_thr:.3f}")

## Stacking / Blending

In [ ]:
stack_cols = [oof_tfidf, oof_xgb]
stack_test_cols = [test_tfidf[:, 1], test_xgb]
labels_stack = ["tfidf", "xgb"]

if 'transformer_oof' in globals():
    stack_cols.append(transformer_oof)
    stack_test_cols.append(transformer_test)
    labels_stack.append("transformer")

stack_X = np.vstack(stack_cols).T
stack_test = np.vstack(stack_test_cols).T

meta = LogisticRegression(max_iter=500, C=1.5)
meta.fit(stack_X, labels)

meta_oof = meta.predict_proba(stack_X)[:, 1]
meta_test = meta.predict_proba(stack_test)[:, 1]

best_meta_thr, best_meta_acc = 0.5, 0
for thr in thr_grid:
    acc = accuracy_score(labels, (meta_oof >= thr).astype(int))
    if acc > best_meta_acc:
        best_meta_acc, best_meta_thr = acc, thr

print(f"Meta stack acc={best_meta_acc:.4f} @ thr={best_meta_thr:.3f}")
print(f"Meta coefficients ({labels_stack}): {meta.coef_[0]}")

calibrator = CalibratedClassifierCV(estimator=LogisticRegression(max_iter=300), method="isotonic", cv=3)
calibrator.fit(stack_X, labels)
meta_calib_oof = calibrator.predict_proba(stack_X)[:, 1]
meta_calib_test = calibrator.predict_proba(stack_test)[:, 1]

best_calib_thr, best_calib_acc = 0.5, 0
for thr in thr_grid:
    acc = accuracy_score(labels, (meta_calib_oof >= thr).astype(int))
    if acc > best_calib_acc:
        best_calib_acc, best_calib_thr = acc, thr
print(f"Meta calib acc={best_calib_acc:.4f} @ thr={best_calib_thr:.3f}")

## Generation des submissions

In [ ]:
test_ids = test_df["challenge_id"].astype(int).to_numpy()

submissions = {}

pred_meta = (meta_calib_test >= best_calib_thr).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": pred_meta}).to_csv(
    SUBMISSION_DIR / "submission_meta_v2.csv", index=False
)
submissions["submission_meta_v2.csv"] = best_calib_thr

pred_meta_raw = (meta_test >= best_meta_thr).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": pred_meta_raw}).to_csv(
    SUBMISSION_DIR / "submission_meta_raw_v2.csv", index=False
)
submissions["submission_meta_raw_v2.csv"] = best_meta_thr

if 'transformer_test' in globals():
    for thr in [0.5, best_tf_thr]:
        pred_tf = (transformer_test >= thr).astype(int)
        name = f"submission_tf_v2_thr{thr:.2f}.csv"
        pd.DataFrame({"ID": test_ids, "Prediction": pred_tf}).to_csv(
            SUBMISSION_DIR / name, index=False
        )
        submissions[name] = thr

pred_tfidf = (test_tfidf[:, 1] >= best_tfidf_thr).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": pred_tfidf}).to_csv(
    SUBMISSION_DIR / "submission_tfidf_v2.csv", index=False
)
submissions["submission_tfidf_v2.csv"] = best_tfidf_thr

pred_xgb = (test_xgb >= best_xgb_thr).astype(int)
pd.DataFrame({"ID": test_ids, "Prediction": pred_xgb}).to_csv(
    SUBMISSION_DIR / "submission_xgb_v2.csv", index=False
)
submissions["submission_xgb_v2.csv"] = best_xgb_thr

print("\nSubmissions ecrites:")
for k, v in submissions.items():
    print(f"  {k} (thr={v:.3f})")

## 🔄 Pseudo-labeling (Self-training)

Utilise les prédictions test high-confidence pour augmenter les données d'entraînement.

In [ ]:
# =============================================================================
# 🔄 PSEUDO-LABELING
# =============================================================================
# Réentraîne le modèle en incluant les prédictions test haute confiance

PSEUDO_THRESHOLD_HIGH = 0.90  # Seuil pour label=1
PSEUDO_THRESHOLD_LOW = 0.10   # Seuil pour label=0
PSEUDO_RATIO = 0.3            # Ratio max de pseudo-labels vs train

# Utiliser les prédictions meta calibrées
if 'meta_calib_test' in dir():
    pseudo_probs = meta_calib_test
elif 'transformer_test' in dir():
    pseudo_probs = transformer_test
else:
    pseudo_probs = test_xgb

# Sélectionner les échantillons haute confiance
high_conf_1 = pseudo_probs >= PSEUDO_THRESHOLD_HIGH
high_conf_0 = pseudo_probs <= PSEUDO_THRESHOLD_LOW
high_conf_mask = high_conf_1 | high_conf_0

n_pseudo = int(len(train_df) * PSEUDO_RATIO)
n_available = high_conf_mask.sum()
n_pseudo = min(n_pseudo, n_available)

print(f"📊 Pseudo-labeling stats:")
print(f"   Haute confiance (label=1, prob≥{PSEUDO_THRESHOLD_HIGH}): {high_conf_1.sum():,}")
print(f"   Haute confiance (label=0, prob≤{PSEUDO_THRESHOLD_LOW}): {high_conf_0.sum():,}")
print(f"   Total disponible: {n_available:,}")
print(f"   Utilisation: {n_pseudo:,} ({100*n_pseudo/len(train_df):.1f}% du train)")

if n_pseudo > 1000:
    # Créer pseudo-labels
    pseudo_labels = (pseudo_probs >= 0.5).astype(int)
    
    # Sélectionner aléatoirement parmi les haute confiance
    high_conf_indices = np.where(high_conf_mask)[0]
    np.random.seed(42)
    selected_indices = np.random.choice(high_conf_indices, size=n_pseudo, replace=False)
    
    # Créer le dataset pseudo-labellé
    pseudo_df = test_df.iloc[selected_indices].copy()
    pseudo_df["label"] = pseudo_labels[selected_indices]
    pseudo_df["is_pseudo"] = True
    
    # Combiner avec train
    train_df["is_pseudo"] = False
    train_pseudo_df = pd.concat([train_df, pseudo_df], ignore_index=True)
    
    print(f"✅ Dataset pseudo-labellé: {len(train_pseudo_df):,} échantillons")
    print(f"   Distribution labels pseudo: {pseudo_df['label'].value_counts().to_dict()}")
    
    # Sauvegarder pour référence
    n_pseudo_1 = pseudo_df["label"].sum()
    n_pseudo_0 = len(pseudo_df) - n_pseudo_1
else:
    print("⚠️ Pas assez d'échantillons haute confiance, pseudo-labeling ignoré")

In [ ]:
# =============================================================================
# 🔄 RÉENTRAÎNEMENT AVEC PSEUDO-LABELS (XGBoost seulement pour rapidité)
# =============================================================================

if 'n_pseudo_1' in dir() and n_pseudo > 1000:
    print("\n🚀 Réentraînement XGBoost avec pseudo-labels...")
    
    # Récupérer les features pour les pseudo-labels
    pseudo_indices = selected_indices
    X_pseudo = X_feat_test[pseudo_indices] if 'X_feat_test' in dir() else None
    
    if X_pseudo is not None:
        # Combiner features
        X_train_pseudo = np.vstack([X_feat, X_pseudo])
        y_train_pseudo = np.concatenate([labels, pseudo_labels[pseudo_indices]])
        
        # Créer sample weights (pseudo-labels ont moins de poids)
        weights_pseudo = np.concatenate([
            np.ones(len(labels)),                    # Train original: poids 1.0
            np.ones(len(pseudo_indices)) * 0.5       # Pseudo-labels: poids 0.5
        ])
        
        # Réentraîner XGBoost avec tous les folds
        skf_pseudo = StratifiedKFold(n_splits=CFG.folds, shuffle=True, random_state=42)
        test_xgb_pseudo = np.zeros(len(test_df))
        
        for fold, (tr_idx, va_idx) in enumerate(skf_pseudo.split(X_train_pseudo, y_train_pseudo)):
            clf = XGBClassifier(
                n_estimators=2000,
                max_depth=10,
                learning_rate=0.025,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=1,
                gamma=0.1,
                reg_alpha=0.05,
                reg_lambda=0.2,
                tree_method="hist",
                device="cuda" if torch.cuda.is_available() else "cpu",
                eval_metric="logloss",
                early_stopping_rounds=100,
                random_state=42,
            )
            clf.fit(
                X_train_pseudo[tr_idx], y_train_pseudo[tr_idx],
                sample_weight=weights_pseudo[tr_idx],
                eval_set=[(X_train_pseudo[va_idx], y_train_pseudo[va_idx])],
                verbose=False
            )
            test_xgb_pseudo += clf.predict_proba(X_feat_test)[:, 1]
        
        test_xgb_pseudo /= CFG.folds
        
        # Nouvelle soumission avec pseudo-labels
        pred_xgb_pseudo = (test_xgb_pseudo >= best_xgb_thr).astype(int)
        pd.DataFrame({"ID": test_ids, "Prediction": pred_xgb_pseudo}).to_csv(
            SUBMISSION_DIR / "submission_xgb_pseudo_v2.csv", index=False
        )
        print(f"✅ submission_xgb_pseudo_v2.csv sauvegardée")
        
        # Nouveau stacking avec XGBoost pseudo
        if 'transformer_oof' in dir():
            # Pour le stacking, on utilise les OOF originaux + test pseudo
            stack_test_pseudo = np.vstack([
                test_tfidf[:, 1], 
                test_xgb_pseudo,  # Remplace test_xgb
                transformer_test
            ]).T
            
            meta_pseudo_test = meta.predict_proba(stack_test_pseudo)[:, 1]
            pred_meta_pseudo = (meta_pseudo_test >= best_meta_thr).astype(int)
            
            pd.DataFrame({"ID": test_ids, "Prediction": pred_meta_pseudo}).to_csv(
                SUBMISSION_DIR / "submission_meta_pseudo_v2.csv", index=False
            )
            print(f"✅ submission_meta_pseudo_v2.csv sauvegardée")
        
        np.save(MODEL_DIR / "xgb_pseudo_test.npy", test_xgb_pseudo)
        clean_memory()
else:
    print("⏭️ Pseudo-labeling non effectué (pas assez de données haute confiance)")

## 📝 Notes et Configuration H100 ULTIMATE

### 🚀 Optimisations H100 appliquées:

| Paramètre | Avant (RTX 4000) | Après (H100) | Gain |
|-----------|------------------|--------------|------|
| **Modèle** | twitter-xlm-roberta-base (278M) | **xlm-roberta-large (560M)** | +2x params |
| **Batch size** | 8 × 2 = 16 | **32 × 1 = 32** | +2x |
| **MAX_LEN** | 224 | **256** | +14% contexte |
| **Seeds** | 1 | **4** (42, 1234, 2024, 0) | -variance |
| **Epochs** | 4 | **5** | +25% |
| **Workers** | 4 | **8** | +2x IO |
| **Precision** | fp16 | **bf16** | +stabilité |
| **XGB arbres** | 2000 | **3000** | +50% |

### 📊 Pourquoi xlm-roberta-large plutôt que twitter-xlm-roberta-base?

**Malheureusement, cardiffnlp n'a PAS de version "large" de twitter-xlm-roberta.**

Options disponibles:
1. `cardiffnlp/twitter-xlm-roberta-base` (278M) - Spécialisé Twitter mais BASE seulement
2. `xlm-roberta-large` (560M) - ✅ **CHOISI** - Plus de capacité, excellent sur le français
3. `camembert/camembert-large` (335M) - Français natif, mais pas Twitter

**xlm-roberta-large** est le meilleur choix car:
- 2x plus de paramètres = meilleure représentation
- Pré-entraîné sur 2.5TB incluant beaucoup de français
- La spécialisation Twitter est moins importante que la capacité du modèle
- Le preprocessing Twitter (@user, http) compense l'absence de pré-entraînement spécifique

### 🧠 Architecture Custom optimisée:

```
xlm-roberta-large (24 layers, 1024 hidden)
    │
    ├── Mean Pooling ──────────┐
    │                          │
    └── Attention Pooling ─────┤
                               │
                         [concat 2048d]
                               │
                         LayerNorm
                               │
                         Linear(2048 → 1024)
                               │
                         GELU + Dropout
                               │
                         Linear(1024 → 2)
                               │
                    Multi-Sample Dropout (×5)
```

### ⏱️ Temps d'exécution estimé (H100):

| Étape | Temps |
|-------|-------|
| Transformer (4 seeds × 5 folds × 5 epochs) | ~2-3h |
| TF-IDF + XGBoost + Stacking | ~20min |
| Pseudo-labeling | ~10min |
| **Total** | **~3-4h** |

### 📈 Performance attendue:

| Méthode | CV Accuracy | Kaggle LB (estimé) |
|---------|------------|-------------------|
| xlm-roberta-large seul | ~87-88% | ~86-87% |
| + XGBoost features | ~88% | ~87% |
| + TF-IDF ensemble | ~88-89% | ~87-88% |
| + Pseudo-labeling | ~89% | ~88-89% |

### 🔄 Alternative si besoin de speed:

Si le temps est limité, vous pouvez revenir à la config rapide:
```python
CFG.model_name = "cardiffnlp/twitter-xlm-roberta-base"
CFG.batch_size = 64  # Très rapide sur H100
CFG.seeds = (42,)
CFG.epochs = 4
# → ~30min total, accuracy ~85-86%
```

### 📋 Ordre de soumission recommandé:
1. 🥇 `submission_meta_pseudo_v2.csv` (stack complet + pseudo-labels)
2. 🥈 `submission_meta_v2.csv` (stack calibré isotonique)
3. 🥉 `submission_tf_v2_thr*.csv` (xlm-roberta-large seul)
4. 🏅 `submission_xgb_pseudo_v2.csv` (XGBoost + pseudo-labels)
5. 🏅 `submission_tfidf_v2.csv` (baseline TF-IDF robuste)